# 03 — Topic Modeling Experiments
NMF vs LDA, n-topic sweep, evaluation metrics, and visualisation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from src.data.make_dataset import load_processed
from src.features.vectorize import (
    build_vectorizer, fit_transform_corpus, get_feature_names, load_vectorizer
)
from src.models.train import (
    build_nmf, build_lda, build_model, train_model,
    get_top_words, save_topics, save_model
)
from src.models.evaluate import (
    evaluate_model, sweep_n_topics, topic_diversity
)
from src.visualization.plots import (
    plot_topic_top_words, plot_topic_distribution_heatmap,
    plot_dominant_topic_counts, plot_sweep_results, plot_wordcloud
)
from src.utils import load_config, set_seed, path_for

cfg = load_config()
set_seed(cfg['random_state'])
df = load_processed()
corpus = df['clean_text'].tolist()
print(f'{len(corpus):,} documents')

## 1. Vectorise Corpus

In [ ]:
try:
    vec = load_vectorizer()
    from src.features.vectorize import transform_corpus
    dtm = transform_corpus(vec, corpus)
except FileNotFoundError:
    vec = build_vectorizer('tfidf', max_features=cfg['max_features'],
                           max_df=cfg['max_df'], min_df=cfg['min_df'])
    vec, dtm = fit_transform_corpus(vec, corpus)

feature_names = get_feature_names(vec)
print('DTM shape:', dtm.shape)

## 2. Train NMF

In [ ]:
nmf = build_nmf(n_topics=cfg['n_topics'], random_state=cfg['random_state'])
train_model(nmf, dtm)

nmf_topics = get_top_words(nmf, feature_names, cfg['n_top_words'])
for t in nmf_topics:
    print(f"[{t['label']}]  {t['top_words']}")

## 3. Train LDA (Count DTM)

In [ ]:
count_vec = build_vectorizer('count', max_features=cfg['max_features'],
                              max_df=cfg['max_df'], min_df=cfg['min_df'])
count_vec, dtm_count = fit_transform_corpus(count_vec, corpus)

lda = build_lda(n_topics=cfg['n_topics'], random_state=cfg['random_state'])
train_model(lda, dtm_count)

lda_topics = get_top_words(lda, get_feature_names(count_vec), cfg['n_top_words'])
for t in lda_topics[:5]:
    print(f"[{t['label']}]  {t['top_words']}")

## 4. Evaluation

In [ ]:
nmf_metrics = evaluate_model(nmf, dtm, feature_names)
lda_metrics = evaluate_model(lda, dtm_count, get_feature_names(count_vec))

metrics_df = pd.DataFrame([nmf_metrics, lda_metrics])
print(metrics_df.to_string(index=False))

## 5. N-topics Sweep (NMF)

In [ ]:
topic_range = [5, 8, 10, 15, 20]
sweep_df = sweep_n_topics(
    model_builder=lambda n: build_nmf(n_topics=n, random_state=42),
    dtm=dtm,
    feature_names=feature_names,
    topic_range=topic_range,
)
print(sweep_df)

In [ ]:
fig = plot_sweep_results(
    sweep_df,
    save_path='../reports/figures/sweep_results.png'
)
plt.show()

## 6. Topic Word Visualisation

In [ ]:
fig = plot_topic_top_words(
    nmf_topics,
    n_top=10,
    save_path='../reports/figures/nmf_topic_words.png'
)
plt.show()

## 7. Heatmap — document × topic

In [ ]:
W = nmf.transform(dtm)     # document-topic matrix
fig = plot_topic_distribution_heatmap(
    W, save_path='../reports/figures/topic_heatmap.png'
)
plt.show()

## 8. Dominant Topic Distribution

In [ ]:
dominant = W.argmax(axis=1).tolist()
fig = plot_dominant_topic_counts(
    dominant, cfg['n_topics'],
    save_path='../reports/figures/dominant_topic_counts.png'
)
plt.show()

## 9. Word Clouds per Topic

In [ ]:
for t in nmf_topics[:3]:
    fig = plot_wordcloud(
        t['words'], t['weights'],
        title=t['label'],
        save_path=f"../reports/figures/wordcloud_{t['topic_id']}.png"
    )
    plt.show()

## 10. Save Best Model

In [ ]:
save_model(nmf, 'nmf_model.joblib')
save_model(lda, 'lda_model.joblib')
save_topics(nmf_topics, 'topics.csv')
print('Models and topics saved.')